In [ ]:
import os
path = "/..."
os.chdir(path + "/llm-introspection-main")

In [ ]:
import textattack
from huggingface_hub import login
login(token="...")

In [ ]:
import sys  
sys.path.append(path + "/wrapper")
from wrapper import TextAttackWrapper

Load the Dataset

In [ ]:
os.chdir(path + '/eraserbenchmark-master')
from rationale_benchmark.utils import load_documents, annotations_from_jsonl
movies_data_root = path + '/eraserbenchmark-master/data/movies'
movies_documents = load_documents(movies_data_root)
movies = annotations_from_jsonl(os.path.join(movies_data_root, 'test.jsonl'))

# Llama

In [9]:
from transformers import LlamaForCausalLM, AutoTokenizer
import torch

#model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
model_name = "meta-llama/Llama-3.2-1B-Instruct"
model = LlamaForCausalLM.from_pretrained(model_name,device_map="auto",torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True,padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

In [10]:
wrapper = TextAttackWrapper(model_family="llama", model=model, tokenizer=tokenizer, task='sentiment', batch_size=16)

In [ ]:
import pandas as pd
from textattack.datasets import Dataset

os.chdir(path + '/eraserbenchmark-master/movies_dataset_builder/my_dataset')
df = pd.read_csv("test.csv")

data = [] 

for row in df.iterrows():
    text = row[1][0]
    res = wrapper([text])
    data.append(([text], int(res[0][1] > res[0][0])))
       
marked_dataset = Dataset(data)
num_examples = len(marked_dataset)

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:32'
os.chdir(path)

attack = TextFoolerJin2019

log_filename = f"movie_attacks/Llama-3.2-3B-Instruct_{attack.__name__}.csv"
attack_args = textattack.AttackArgs(num_examples=num_examples, log_to_csv=log_filename, disable_stdout=True, parallel=False)
attacker = textattack.Attacker(attack.build(wrapper), marked_dataset, attack_args)
attacker.attack_dataset()
del attacker
torch.cuda.empty_cache()

textattack: Unknown if model of class <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'> compatible with goal function <class 'textattack.goal_functions.classification.untargeted_classification.UntargetedClassification'>.
textattack: Logging to CSV at path movie_attacks/Llama-3.2-3B-Instruct_TextFoolerJin2019.csv


Attack(
  (search_method): GreedyWordSwapWIR(
    (wir_method):  delete
  )
  (goal_function):  UntargetedClassification
  (transformation):  WordSwapEmbedding(
    (max_candidates):  50
    (embedding):  WordEmbedding
  )
  (constraints): 
    (0): WordEmbeddingDistance(
        (embedding):  WordEmbedding
        (min_cos_sim):  0.5
        (cased):  False
        (include_unknown_words):  True
        (compare_against_original):  True
      )
    (1): PartOfSpeech(
        (tagger_type):  nltk
        (tagset):  universal
        (allow_verb_noun_swap):  True
        (compare_against_original):  True
      )
    (2): UniversalSentenceEncoder(
        (metric):  angular
        (threshold):  0.840845057
        (window_size):  15
        (skip_text_shorter_than_window):  True
        (compare_against_original):  False
      )
    (3): RepeatModification
    (4): StopwordModification
    (5): InputColumnModification(
        (matching_column_labels):  ['premise', 'hypothesis']
       

/venv/main/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/venv/main/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
I0000 00:00:1760887152.259504    9728 gpu_process_state.cc:208] Using CUDA malloc Async allocator for GPU: 0
I0000 00:00:1760887152.260854    9728 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 61858 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:c7:00.0, compute capability: 8.0
[Succeeded / Failed / Skipped / Total] 1 / 2 / 0 / 3:   2

# Qwen

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name,device_map="cuda",torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True,padding_side='left')
tokenizer.pad_token = tokenizer.eos_token

wrapper = TextAttackWrapper(model_family="qwen", model=model, tokenizer=tokenizer, task='sentiment', batch_size=16)

In [ ]:
import pandas as pd
from textattack.datasets import Dataset

os.chdir(path + '/eraserbenchmark-master/movies_dataset_builder/my_dataset')
df = pd.read_csv("test.csv")

data = [] 

for row in df.iterrows():
    text = row[1][0]
    res = wrapper([text])
    if res[0][1] > res[0][0]:
        data.append((text,1))
    else:
        data.append((text,0))
        
marked_dataset = Dataset(data)
num_examples = len(marked_dataset)

In [ ]:
from textattack.attack_recipes import TextFoolerJin2019
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:32'
os.chdir(path)

attack_recipe = TextFoolerJin2019

log_filename = f"movie_attacks/Qwen-7B_{attack.__name__}.csv"
attack_args = textattack.AttackArgs(num_examples=num_examples, log_to_csv=log_filename, disable_stdout=True, parallel=False)
attacker = textattack.Attacker(attack.build(wrapper), marked_dataset, attack_args)
attacker.attack_dataset()
del attacker
torch.cuda.empty_cache()